In [7]:
import os
import json
import pandas as pd

In [8]:
def consolidar_incremental():
    # 1. Configuración de rutas para Jupyter
    path_actual = os.getcwd() 
    output_name = "dataset_millonarios_consolidado.csv"
    output_path = os.path.join(path_actual, output_name)
    
    lista_registros_nuevos = []
    partidos_procesados = set()

    # 2. Cargar datos existentes si el archivo ya fue creado
    if os.path.exists(output_path):
        # Cargamos el CSV existente. Los nulos se cargan como NaN automáticamente.
        df_existente = pd.read_csv(output_path, encoding='utf-16')
        if not df_existente.empty:
            # Identificador único para evitar duplicados
            partidos_procesados = set(zip(df_existente['fecha'].astype(str), df_existente['rival'].astype(str)))
            print(f"Dataset cargado: {len(partidos_procesados)} partidos ya existen en el CSV.")
    else:
        df_existente = pd.DataFrame()
        print("Iniciando dataset desde cero.")

    # 3. Función auxiliar para manejar nulos
    def clean_val(diccionario, llave):
        val = diccionario.get(llave)
        # Si el valor es None o un string vacío, devolvemos NaN de numpy
        if val is None or val == "":
            return np.nan
        return val

    # 4. Escaneo de subcarpetas en el directorio actual
    for nombre_item in os.listdir(path_actual):
        ruta_subcarpeta = os.path.join(path_actual, nombre_item)
        
        if os.path.isdir(ruta_subcarpeta) and "Millonarios" in nombre_item:
            archivos_json = [f for f in os.listdir(ruta_subcarpeta) if f.endswith(".json")]
            
            for archivo in archivos_json:
                ruta_archivo = os.path.join(ruta_subcarpeta, archivo)
                
                with open(ruta_archivo, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                meta = data.get('metadata', {})
                fecha = str(meta.get("fecha"))
                rival = str(meta.get("rival"))

                # Evitar duplicar si ya lo procesamos antes
                if (fecha, rival) in partidos_procesados:
                    continue 
                
                jugadores = data.get('jugadores') or data.get('plantilla', [])
                for p in jugadores:
                    rend = p.get('rendimiento', {})
                    pases = rend.get('pases', {})
                    defensa = rend.get('defensa', {})
                    duelos = rend.get('duelos', {})
                    faltas = rend.get('faltas', {})
                    tarjetas = rend.get('tarjetas', {})
                    
                    registro = {
                        "fecha": fecha,
                        "campeonato": meta.get("campeonato") or meta.get("liga"),
                        "rival": rival,
                        "condicion": meta.get("condicion"),
                        "resultado": meta.get("resultado"),
                        "jugador": p.get("nombre"),
                        "posicion": p.get("posicion"),
                        "minutos": clean_val(p, "minutos"),
                        "calificacion": pd.to_numeric(p.get("calificacion"), errors='coerce'),
                        "titular": p.get("titular"),
                        
                        # Ataque
                        "goles": clean_val(rend, "goles"),
                        "asistencias": clean_val(rend, "asistencias"),
                        "remates_totales": clean_val(rend, "remates_totales"),
                        "remates_al_arco": clean_val(rend, "remates_al_arco"),
                        
                        # Pases
                        "pases_totales": clean_val(pases, "totales"),
                        "pases_precision": clean_val(pases, "precision"),
                        
                        # Defensa
                        "entradas": clean_val(defensa, "entradas"),
                        "intercepciones": clean_val(defensa, "intercepciones"),
                        "despejes": clean_val(defensa, "despejes"),
                        
                        # Duelos
                        "duelos_totales": clean_val(duelos, "totales"),
                        "duelos_ganados": clean_val(duelos, "ganados"),
                        
                        # Faltas
                        "faltas_cometidas": clean_val(faltas, "cometidas"),
                        "faltas_recibidas": clean_val(faltas, "recibidas"),
                        
                        # Tarjetas
                        "amarillas": clean_val(tarjetas, "amarilla"),
                        "rojas": clean_val(tarjetas, "roja")
                    }
                    lista_registros_nuevos.append(registro)

    # 5. Consolidación y visualización
    if lista_registros_nuevos:
        df_nuevo = pd.DataFrame(lista_registros_nuevos)
        df_final = pd.concat([df_existente, df_nuevo], ignore_index=True)
        
        # Guardar (na_rep='' asegura que los NaN se vean como celdas vacías en Excel/BI)
        df_final.to_csv(output_path, index=False, encoding='utf-16', na_rep='')
        print(f"✓ Éxito: Se añadieron {len(df_nuevo)} registros nuevos.")
    else:
        df_final = df_existente
        print("No se encontraron partidos nuevos para procesar.")

    # Retornar el dataframe para que Jupyter lo muestre
    return df_final


In [9]:
# Ejecutar y mostrar las primeras 5 filas
df_millos = consolidar_incremental()
df_millos.head()

Iniciando dataset desde cero.
✓ Éxito: Se añadieron 3110 registros nuevos.


,fecha,campeonato,rival,condicion,resultado,jugador,posicion,minutos,calificacion,titular,...,pases_precision,entradas,intercepciones,despejes,duelos_totales,duelos_ganados,faltas_cometidas,faltas_recibidas,amarillas,rojas
0,2022-01-22,Primera A,Deportivo Pasto,Visitante,0 - 1,Álvaro Montero,G,90,7.5,True,...,10%,0,0,0,3,2,1,0,1,0
1,2022-01-22,Primera A,Deportivo Pasto,Visitante,0 - 1,Elvis Perlaza,D,90,7.3,True,...,49%,2,1,0,7,5,1,1,0,0
2,2022-01-22,Primera A,Deportivo Pasto,Visitante,0 - 1,Andrés Llinás,D,90,7.5,True,...,63%,3,3,0,8,6,0,0,0,0
3,2022-01-22,Primera A,Deportivo Pasto,Visitante,0 - 1,Juan Pablo Vargas,D,90,7.0,True,...,52%,1,0,0,9,5,3,0,0,0
4,2022-01-22,Primera A,Deportivo Pasto,Visitante,0 - 1,Omar Bertel,D,90,7.7,True,...,54%,5,0,0,9,6,2,0,0,0
